In [ ]:
import numpy as np
import time
from pynq import allocate, Overlay
from scipy.signal import convolve2d

# ==============================================================
# 1.) INITIALIZATION
# ==============================================================
ol = Overlay("accel_v2.bit")
dma = ol.axi_dma_0

IMG_SIZE = 28
OUT_DIM = 13   
CHANNELS = 16

# From 
hex_weights = [
    "0011036C0797F7DDF842F41E013007E9FFD6", "FEE507F3F9E802230651F79B02E8FDD5FC7D",
    "FF1BF2A6F8FA06F20042F9F60A5001A2FFDA", "F8EEFF85FD1EFEA0F8E0F8DB08060790017C",
    "017FFDF802C003E504D0FE400493FC14F957", "00FA0346015A02D2FA720411F94E04650299",
    "F939F981F603F95C01BB041B074703F105EF", "FBEAFD9B0437046F0098FDEA0100FA7800CF",
    "FCB90611FB9AF9FE06C7FE67FF2500E800A3", "FD71071C014DF8DD02A00789F7B2FADD04D7",
    "0380FD00FF380470F7FA032E04F1F10608B0", "02850602FF94FF4D06670222F8CFF7AB02A9",
    "FA7DFA250061FCF201CA048E054904CBFFE6", "0567007EFAF0FC53036103B6FE3AFC0302CD",
    "009D09350837FA2DFEA101AFF938F8FBF4E7", "FB2C0373FF11FAE2013FFFEFFAE7FD61F959"
]

# ==============================================================
# 2.) DATA PREPARATION
# ==============================================================
test_img = np.zeros((28, 28), dtype=np.int16)
test_img[5:23, 13:15] = 4096  
test_img[5:7, 5:23] = 4096   

in_buf = allocate(shape=(784,), dtype=np.int16)
out_buf = allocate(shape=(OUT_DIM, OUT_DIM, CHANNELS), dtype=np.int16)
in_buf[:] = test_img.flatten()

# ==============================================================
# 3.) HARDWARE EXECUTION & LATENCY
# ==============================================================
print("--- FPGA Execution ---")
start_fpga = time.perf_counter()

dma.recvchannel.transfer(out_buf)
dma.sendchannel.transfer(in_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()

end_fpga = time.perf_counter()
fpga_latency = (end_fpga - start_fpga) * 1000
print(f"FPGA Latency: {fpga_latency:.4f} ms")

# ==============================================================
# 4.) CPU REFERENCE & LATENCY
# ==============================================================
def run_cpu_ref(img, hex_list):
    def twos_comp(val):
        return val - 0x10000 if val & 0x8000 else val
    
    golden_out = np.zeros((OUT_DIM, OUT_DIM, CHANNELS), dtype=np.int16)
    for c in range(CHANNELS):
        chunks = [hex_list[c][i:i+4] for i in range(0, len(hex_list[c]), 4)]
        chunks.reverse()
        kernel = np.array([twos_comp(int(x, 16)) for x in chunks]).reshape(3,3)
        res = convolve2d(img, kernel, mode='valid')
        res = np.maximum(0, res >> 12).astype(np.int16)
        h, w = res.shape
        pooled = res[:h-h%2, :w-w%2].reshape(h//2, 2, w//2, 2).max(axis=(1, 3))
        golden_out[:, :, c] = pooled
    return golden_out

print("\n--- CPU Execution ---")
start_cpu = time.perf_counter()
expected = run_cpu_ref(test_img, hex_weights)
end_cpu = time.perf_counter()
cpu_latency = (end_cpu - start_cpu) * 1000
print(f"CPU Latency: {cpu_latency:.4f} ms")

# Speedup Calculation
print(f"\nSpeedup: {cpu_latency / fpga_latency:.2f}x")

# ==============================================================
# 5.) FULL COMPARISON & DATA DUMP
# ==============================================================
match = np.array_equal(out_buf, expected)
print(f"\nResult Match: {match}")

def print_channel_map(data, channel_idx, label):
    print(f"\n{label} - Channel {channel_idx} Map ({OUT_DIM}x{OUT_DIM}):")
    # Iterate through rows
    for r in range(OUT_DIM):
        row_str = " ".join(f"{data[r, c, channel_idx]:4d}" for c in range(OUT_DIM))
        print(row_str)

# Print the full maps for the first 2 channels to inspect visually
for i in range(2):
    print_channel_map(out_buf, i, "FPGA")
    print_channel_map(expected, i, "CPU")

if not match:
    diff_count = np.sum(out_buf != expected)
    print(f"\nTotal Mismatched Pixels: {diff_count} / {out_buf.size}")
    
# ==============================================================
# BATCH THROUGHPUT MEASUREMENT
# ==============================================================
BATCH_SIZE = 1000 

# Allocate large buffers for the batch
batch_in = allocate(shape=(BATCH_SIZE, 784), dtype=np.int16)
batch_out = allocate(shape=(BATCH_SIZE, OUT_DIM, OUT_DIM, CHANNELS), dtype=np.int16)

# Fill with test data
batch_in[:] = test_img.flatten() 

print(f"\n--- Measuring Throughput (Batch Size: {BATCH_SIZE}) ---")

start_tp = time.perf_counter()

# Loop through the batch
for i in range(BATCH_SIZE):
    dma.recvchannel.transfer(batch_out[i])
    dma.sendchannel.transfer(batch_in[i])
    dma.sendchannel.wait()
    dma.recvchannel.wait()

end_tp = time.perf_counter()

total_time = end_tp - start_tp
fps = BATCH_SIZE / total_time
throughput_latency = (total_time / BATCH_SIZE) * 1000

print(f"Total Time for {BATCH_SIZE} images: {total_time:.4f} seconds")
print(f"Throughput: {fps:.2f} Images Per Second (FPS)")
print(f"Average Latency in Batch: {throughput_latency:.4f} ms")

--- FPGA Execution ---
FPGA Latency: 3.1951 ms

--- CPU Execution ---
CPU Latency: 27.3537 ms

Speedup: 8.56x

Result Match: True

FPGA - Channel 0 Map (13x13):
   0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0 2287 2287 2287 2287 2287 2287 2287 2287 2329  304    0
   0    0    0    0    0    0    0    0    0    0    0    0    0
   0 1943 2836 2836 2836 2836 1100 2836 2836 2836 2836   17    0
   0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0    0
   0    0    0    0    0 1943 2819   17    0    0    0    0    0
   0    0    0    0    0    0    0    0    0    0    0    0